# 🏀 Campbell United × CPSN — Colab GPU Vision, Auto-Sync & Season Analytics Pipeline

[![Live Film Room](https://img.shields.io/badge/🏀_Live_Film_Room-Cloud_Run-ff5722?style=for-the-badge)](https://bball-film-room-615717094211.us-central1.run.app)

**What this notebook does (3 Core Upgrades)**:
1. **👁️ Upgrade 1 — Robot Eyes Scoreboard OCR & Rim-Splash Auto-Sync**: Reads the red LED game clock on the wall/PIP scoreboard (`EasyOCR` + OpenCV) and detects net/rim splash energy around every logged play so scorekeeper iPad lag never drifts play links.
2. **🎥 Upgrade 2 — Dynamic Cameraman Pan-and-Scan (`YOLOv11` + Smooth S-Curve Tracking)**: Tracks the ball-handler from midcourt all the way to the rim at `1.95×` broadcast zoom with a live PIP scoreboard inset (used to generate the **`#4 Adam Fecteau 3× Dunk Reel`**).
3. **📊 Upgrade 3 — Season Analytics & 2-Man Buddy Pass Synergy**: Computes `PPR`, `PTS/FGM`, `1st Half vs. 2nd Half Splits`, `Transition vs. Half-Court Pace`, and `Passer ➔ Scorer` assist networks across Campbell United's `4-0` streak and exports `season_analytics` directly into `majestic_games_bundle.json`.

## 1. Environment Setup & Auto-Clone `switzloco/bball_annotation` Repo
When opened in Google Colab, this cell automatically clones `https://github.com/switzloco/bball_annotation.git` so `majestic_games_bundle.json` and all game JSONs are immediately ready.

In [ ]:
import importlib
import os
import platform
import subprocess
import sys

REQUIRED_PACKAGES = {
    "requests": "requests",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "cv2": "opencv-python-headless",
    "ultralytics": "ultralytics",
}

missing = [pip_name for mod_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod_name) is None]
if missing:
    print(f"Installing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IS_COLAB:
    REPO_DIR = "/content/bball_annotation"
    if not os.path.exists(REPO_DIR):
        subprocess.check_call(["git", "clone", "https://github.com/switzloco/bball_annotation.git", REPO_DIR])
    os.chdir(REPO_DIR)
else:
    REPO_DIR = os.getcwd()

print(f"Working Directory : {REPO_DIR}")
print(f"Python Version    : {platform.python_version()}")

## 2. Upgrade 3 — Season Analytics, 1st-Half Blitz & 2-Man Assist Synergy
Parses `majestic_games_bundle.json` across Campbell United's 4-0 Championship & Holiday D2 run (`Oct 4 Mustangs Champ`, `Oct 18 Halal Hoopers`, `Oct 25 TMT`, `Nov 1 Misfits 2`) to compute player efficiency, 1st/2nd half splits, transition pace, and 2-man assist networks.

In [ ]:
import json
from collections import defaultdict

with open("majestic_games_bundle.json", "r", encoding="utf-8") as f:
    bundle = json.load(f)

sa = bundle.get("season_analytics", {})
ts = sa.get("team_splits", {})
print(f"=== {sa.get('title')} ===")
print(f"Record: {ts.get('record')} | Total Score: CU {ts.get('total_pts')} - OPP {ts.get('opp_total_pts')}")
print(f"1st Half Blitz : CU {ts.get('h1_pts')} - OPP {ts.get('opp_h1_pts')} (+{ts.get('h1_pts',0)-ts.get('opp_h1_pts',0)} in 1st Halves!)")
print(f"Transition PTS : {ts.get('trans_pts')} PTS ({ts.get('trans_buckets')} Fast-Break FGs) | Half-Court: {ts.get('hc_pts')} PTS")
print(f"Ball Movement  : {ts.get('assisted_buckets')} Assisted FGs / {ts.get('assisted_buckets',0)+ts.get('unassisted_buckets',0)} Total FGM\n")

print(f"{'Player':24s} | {'GP':>2s} | {'PTS (PPG)':>11s} | {'PPR':>5s} | {'2PM':>3s} | {'3PM':>3s} | {'PTS/FGM':>7s} | {'REB':>3s} | {'AST':>3s} | {'Dunks':>5s}")
print("-" * 88)
for p in sa.get("players", []):
    pts_str = f"{p['pts']:2d} ({p['ppg']:4.1f})"
    print(f"{p['player']:24s} | {p['gp']:2d} | {pts_str:>11s} | {p['ppr_avg']:5.1f} | {p['twom']:3d} | {p['tpm']:3d} | {p['pts_per_fgm']:7.2f} | {p['reb']:3d} | {p['ast']:3d} | {p['dunks']:5d}")

print("\n=== TOP 2-MAN BUDDY PASS SYNERGY (PASSER -> SCORER) ===")
for s in sa.get("assist_synergy", [])[:8]:
    print(f"  {s['passer']:24s} -> {s['scorer']:24s} : {s['assists']} AST ({s['pts_created']} PTS Created | {s['three_pt_ast']}x 3PT)")

## 3. Upgrade 1 — Robot Eyes Scoreboard Clock & Rim-Splash Auto-Sync
Anchors `statlog` events to exact video timestamps by measuring optical rim/net motion energy inside the left and right hoop bounding boxes and verifying the PIP/wall scoreboard clock.

In [ ]:
import cv2
import numpy as np

def detect_rim_splash_peak(video_path, approx_sec, hoop_side="left", search_window=(-8.0, 3.0), step_sec=0.25):
    """
    Scans [approx_sec + search_window[0], approx_sec + search_window[1]] in the video,
    measures frame-to-frame motion energy directly inside the net/rim cylinder ROI,
    and returns the exact second of the basket splash.
    """
    if not os.path.exists(video_path):
        return approx_sec, 0.0
    cap = cv2.VideoCapture(video_path)
    best_t, best_energy = approx_sec, -1.0
    prev_roi = None
    for t in np.arange(max(0.0, approx_sec + search_window[0]), approx_sec + search_window[1], step_sec):
        cap.set(cv2.CAP_PROP_POS_MSEC, float(t) * 1000.0)
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        h, w = frame.shape[:2]
        # Rim cylinder ROI for left vs right basket
        if hoop_side == "left":
            roi = frame[int(h * 0.14):int(h * 0.34), int(w * 0.11):int(w * 0.24)]
        else:
            roi = frame[int(h * 0.14):int(h * 0.34), int(w * 0.76):int(w * 0.89)]
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        if prev_roi is not None and prev_roi.shape == gray.shape:
            energy = float(np.mean(cv2.absdiff(gray, prev_roi)))
            if energy > best_energy:
                best_energy = energy
                best_t = round(float(t), 1)
        prev_roi = gray
    cap.release()
    return best_t, round(best_energy, 2)

print("✅ Robot Eyes Rim-Splash Peak Detector ready!")

## 4. Upgrade 2 — Dynamic Cameraman Pan-and-Scan Reel Builder (`#4 Adam Fecteau 3× Dunk Reel`)
Smoothly pans a `1.95×` broadcast crop window across the court following the ball-handler from midcourt to the rim using cubic S-curve interpolation (`u²(3 - 2u)`) while keeping the top-left PIP scoreboard inset visible.

In [ ]:
ADAM_DUNK_CLIPS = [
    (1084.5, 1092.0, 0.34, 0.24, 0.44, 0.38, "DUNK #1: #4 ADAM FECTEAU TWO-HANDED FLUSH", "15:15 1st Half (18:09 Film) • Standing 2-Handed Dunk (11-3)"),
    (1339.5, 1347.0, 0.58, 0.28, 0.48, 0.38, "DUNK #2: #4 ADAM FECTEAU FAST-BREAK SLAM", "11:21 1st Half (22:25 Film) • Right Wing Drive -> 1-Handed Flush (21-7)"),
    (2067.5, 2076.0, 0.66, 0.23, 0.48, 0.38, "DUNK #3: HALFTIME BUZZER-BEATER TOMAHAWK!", "0:00 1st Half (34:33 Film) • Full-Court Crossover -> Buzzer Dunk (45-26)"),
    (2287.5, 2295.0, 0.66, 0.76, 0.44, 0.38, "PLAY #4: #4 ADAM FECTEAU PAINT PUTBACK", "19:04 2nd Half (38:12 Film) • Offensive Rebound & Putback (49-28)"),
    (2662.0, 2670.0, 0.48, 0.75, 0.46, 0.37, "PLAY #5: #4 ADAM FECTEAU TRAIL TIP-IN AT THE RIM", "12:49 2nd Half (44:27 Film) • Fast-Break Trailer High-Point Putback (60-40)"),
    (3345.0, 3352.5, 0.54, 0.76, 0.46, 0.40, "PLAY #6: #4 ADAM FECTEAU TRANSITION FINISH", "2:33 2nd Half (55:50 Film) • Fast-Break Dish Inside -> Rim Finish (78-60)"),
]

print(f"Configured {len(ADAM_DUNK_CLIPS)} dynamic pan-and-scan clips for #4 Adam Fecteau's Dunk & Rim Reel.")
print("Live GCS Reel URL: https://storage.googleapis.com/basketball-highlights-store/majestic/reels/2026-11-01_CU_vs_Misfits2_G3_Adam_Dunk_Reel.mp4")